# 第 3 週實作：監督式學習概念、資料分割與交叉驗證
# Week 3 Lab: Supervised Learning, Data Splitting & Cross-Validation

**學習目標 Learning Objectives:**
1. 使用 `train_test_split` 進行資料分割並視覺化結果
2. 實作 k-Fold 交叉驗證並視覺化每個 Fold 的分割情況
3. 比較 Stratified 與 Non-Stratified 分割的差異
4. 透過多項式回歸觀察偏差-變異權衡 (Bias-Variance Tradeoff)
5. 視覺化過擬合 (Overfitting) 與欠擬合 (Underfitting) 現象

---

In [ ]:
# 匯入所需套件 Import required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    cross_val_score, LeaveOneOut, TimeSeriesSplit
)
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
from sklearn.datasets import make_classification, make_moons

# 設定繪圖風格 Set plotting style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# 設定隨機種子 Set random seed
np.random.seed(42)

print("所有套件匯入成功！ All packages imported successfully!")

## Part 1: `train_test_split` 實作與視覺化

### 1.1 建立範例資料集

我們先建立一個二維分類資料集，方便在散佈圖上視覺化分割結果。

In [ ]:
# 建立二維分類資料集 Create a 2D classification dataset
X, y = make_classification(
    n_samples=300,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    class_sep=1.5,
    random_state=42
)

print(f"資料集形狀 Dataset shape: X={X.shape}, y={y.shape}")
print(f"類別分布 Class distribution: {np.bincount(y)}")
print(f"類別 0: {np.sum(y==0)} 筆, 類別 1: {np.sum(y==1)} 筆")

### 1.2 使用 `train_test_split` 並視覺化結果

觀察分割後，訓練集與測試集在特徵空間中的分布情況。

In [ ]:
# 分割資料：80% 訓練、20% 測試
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"訓練集 Training set: {X_train.shape[0]} 筆")
print(f"測試集 Test set: {X_test.shape[0]} 筆")
print(f"訓練集類別分布: {np.bincount(y_train)}")
print(f"測試集類別分布: {np.bincount(y_test)}")

# 視覺化分割結果 Visualize the split
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 原始資料 Original data
colors = ['#3498db', '#e74c3c']
for cls in [0, 1]:
    mask = y == cls
    axes[0].scatter(X[mask, 0], X[mask, 1], c=colors[cls], 
                    alpha=0.6, edgecolors='white', s=50,
                    label=f'Class {cls}')
axes[0].set_title('Original Dataset\n原始資料集', fontsize=14)
axes[0].legend()
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# 訓練集 Training set
for cls in [0, 1]:
    mask = y_train == cls
    axes[1].scatter(X_train[mask, 0], X_train[mask, 1], c=colors[cls],
                    alpha=0.6, edgecolors='white', s=50,
                    label=f'Class {cls}')
axes[1].set_title(f'Training Set ({X_train.shape[0]} samples)\n訓練集', fontsize=14)
axes[1].legend()
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

# 測試集 Test set
for cls in [0, 1]:
    mask = y_test == cls
    axes[2].scatter(X_test[mask, 0], X_test[mask, 1], c=colors[cls],
                    alpha=0.6, edgecolors='white', s=50,
                    label=f'Class {cls}')
axes[2].set_title(f'Test Set ({X_test.shape[0]} samples)\n測試集', fontsize=14)
axes[2].legend()
axes[2].set_xlabel('Feature 1')
axes[2].set_ylabel('Feature 2')

plt.tight_layout()
plt.suptitle('train_test_split Visualization\ntrain_test_split 視覺化', 
             fontsize=16, y=1.05)
plt.show()

### 1.3 不同 test_size 的影響

嘗試不同的分割比例，觀察訓練集與測試集大小的變化。

In [ ]:
# 比較不同 test_size 的分割結果
test_sizes = [0.1, 0.2, 0.3, 0.4, 0.5]

fig, axes = plt.subplots(1, len(test_sizes), figsize=(20, 4))

for i, ts in enumerate(test_sizes):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=ts, random_state=42, stratify=y
    )
    
    # 繪製訓練集（藍色調）與測試集（紅色調）
    axes[i].scatter(X_tr[:, 0], X_tr[:, 1], c='#3498db', alpha=0.5, 
                    s=20, label=f'Train ({len(X_tr)})')
    axes[i].scatter(X_te[:, 0], X_te[:, 1], c='#e74c3c', alpha=0.7, 
                    s=30, marker='x', label=f'Test ({len(X_te)})')
    axes[i].set_title(f'test_size={ts}\n{len(X_tr)} train / {len(X_te)} test', 
                      fontsize=11)
    axes[i].legend(fontsize=8, loc='upper right')
    axes[i].set_xlabel('Feature 1')
    if i == 0:
        axes[i].set_ylabel('Feature 2')

plt.suptitle('Effect of Different test_size Values\n不同 test_size 的影響', 
             fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

## Part 2: k-Fold 交叉驗證視覺化 k-Fold CV Visualization

### 2.1 視覺化 k-Fold 的分割情況

用顏色區分每個 Fold 中的訓練集與驗證集，直觀理解交叉驗證的原理。

In [ ]:
# 視覺化 5-Fold CV 的分割方式
# Visualize 5-Fold CV split pattern

n_samples = 50  # 使用較小的樣本數以便觀察
n_splits = 5
indices = np.arange(n_samples)

kf = KFold(n_splits=n_splits, shuffle=False)

fig, axes = plt.subplots(n_splits, 1, figsize=(14, 6), sharex=True)

cmap_train = '#3498db'  # 藍色=訓練
cmap_val = '#e74c3c'    # 紅色=驗證

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(indices)):
    ax = axes[fold_idx]
    
    # 繪製每個樣本的顏色條
    colors = np.array([cmap_train] * n_samples)
    colors[val_idx] = cmap_val
    
    for i in range(n_samples):
        ax.barh(0, 1, left=i, color=colors[i], edgecolor='white', linewidth=0.5)
    
    ax.set_yticks([0])
    ax.set_yticklabels([f'Fold {fold_idx + 1}'], fontsize=12)
    ax.set_xlim(0, n_samples)
    ax.set_ylim(-0.5, 0.5)
    
    # 標註驗證集範圍
    val_start, val_end = val_idx[0], val_idx[-1]
    ax.annotate(f'Val: [{val_start}-{val_end}]', 
                xy=((val_start + val_end) / 2, 0.3),
                fontsize=9, ha='center', color='#c0392b', fontweight='bold')

axes[-1].set_xlabel('Sample Index (樣本索引)', fontsize=12)

# 建立圖例 Create legend
train_patch = mpatches.Patch(color=cmap_train, label='Training Set (訓練集)')
val_patch = mpatches.Patch(color=cmap_val, label='Validation Set (驗證集)')
fig.legend(handles=[train_patch, val_patch], loc='upper right', fontsize=11)

plt.suptitle('5-Fold Cross-Validation Split Visualization\n5 折交叉驗證分割視覺化', 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 2.2 k-Fold CV 在特徵空間中的分割

觀察每個 Fold 中驗證集的樣本在特徵空間中的位置。

In [ ]:
# 在特徵空間中視覺化 5-Fold CV
# Visualize 5-Fold CV in feature space

kf = KFold(n_splits=5, shuffle=True, random_state=42)

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X)):
    ax = axes[fold_idx]
    
    # 訓練集（灰色背景）
    ax.scatter(X[train_idx, 0], X[train_idx, 1], 
               c='#bdc3c7', alpha=0.3, s=15, label='Train')
    
    # 驗證集（按類別上色）
    for cls, color in zip([0, 1], ['#3498db', '#e74c3c']):
        mask = y[val_idx] == cls
        ax.scatter(X[val_idx][mask, 0], X[val_idx][mask, 1],
                   c=color, alpha=0.8, s=40, edgecolors='black', linewidth=0.5,
                   label=f'Val class {cls}')
    
    ax.set_title(f'Fold {fold_idx + 1}\nVal: {len(val_idx)} samples', fontsize=11)
    if fold_idx == 0:
        ax.set_ylabel('Feature 2')
    ax.set_xlabel('Feature 1')
    ax.legend(fontsize=7, loc='upper right')

plt.suptitle('5-Fold CV: Validation Samples in Feature Space\n5 折 CV：驗證集在特徵空間的分布', 
             fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

### 2.3 使用 `cross_val_score` 進行交叉驗證

實際使用 scikit-learn 的 `cross_val_score` 進行模型評估。

In [ ]:
# 使用 cross_val_score 進行 k-Fold CV
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42)

# 比較不同 k 值的結果
k_values = [3, 5, 7, 10]
results = {}

for k in k_values:
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy')
    results[k] = scores
    print(f"k={k:2d}: 各 Fold 分數 = {scores.round(4)}")
    print(f"       平均 Mean = {scores.mean():.4f} +/- {scores.std():.4f}")
    print()

# 視覺化不同 k 值的分數分布
fig, ax = plt.subplots(figsize=(10, 5))
positions = range(len(k_values))
bp = ax.boxplot([results[k] for k in k_values], positions=positions,
                patch_artist=True, widths=0.5)

colors_box = ['#3498db', '#2ecc71', '#f39c12', '#9b59b6']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.set_xticklabels([f'k={k}' for k in k_values], fontsize=12)
ax.set_ylabel('Accuracy (準確率)', fontsize=12)
ax.set_title('Cross-Validation Scores for Different k Values\n不同 k 值的交叉驗證分數比較', 
             fontsize=14)
ax.axhline(y=np.mean([results[k].mean() for k in k_values]), 
           color='red', linestyle='--', alpha=0.5, label='Overall Mean')
ax.legend()
plt.tight_layout()
plt.show()

## Part 3: Stratified vs Non-Stratified 比較

### 3.1 建立不平衡資料集

建立一個類別嚴重不平衡的資料集，觀察分層與非分層分割的差異。

In [ ]:
# 建立不平衡資料集 Create an imbalanced dataset
# 90% 類別 0, 10% 類別 1
X_imb, y_imb = make_classification(
    n_samples=200, n_features=2, n_informative=2,
    n_redundant=0, n_clusters_per_class=1,
    weights=[0.9, 0.1],  # 90% vs 10%
    class_sep=1.0, random_state=42
)

print(f"不平衡資料集 Imbalanced dataset:")
print(f"  類別 0: {np.sum(y_imb == 0)} 筆 ({np.mean(y_imb == 0)*100:.1f}%)")
print(f"  類別 1: {np.sum(y_imb == 1)} 筆 ({np.mean(y_imb == 1)*100:.1f}%)")

# 比較 KFold vs StratifiedKFold 中各 Fold 的類別比例
n_splits = 5

kf_normal = KFold(n_splits=n_splits, shuffle=True, random_state=42)
kf_strat = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# 記錄每個 Fold 中類別 1 的比例
normal_ratios = []
strat_ratios = []

for train_idx, val_idx in kf_normal.split(X_imb):
    ratio = np.mean(y_imb[val_idx] == 1)
    normal_ratios.append(ratio)

for train_idx, val_idx in kf_strat.split(X_imb, y_imb):
    ratio = np.mean(y_imb[val_idx] == 1)
    strat_ratios.append(ratio)

# 視覺化比較
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左圖：各 Fold 中少數類比例的比較
x_pos = np.arange(n_splits)
width = 0.35

bars1 = axes[0].bar(x_pos - width/2, normal_ratios, width, 
                     label='KFold (普通)', color='#3498db', alpha=0.7)
bars2 = axes[0].bar(x_pos + width/2, strat_ratios, width, 
                     label='StratifiedKFold (分層)', color='#e74c3c', alpha=0.7)

original_ratio = np.mean(y_imb == 1)
axes[0].axhline(y=original_ratio, color='green', linestyle='--', linewidth=2,
                label=f'Original ratio ({original_ratio:.2%})')

axes[0].set_xlabel('Fold', fontsize=12)
axes[0].set_ylabel('Minority Class Ratio\n少數類比例', fontsize=12)
axes[0].set_title('Class 1 Ratio in Each Fold\n各 Fold 中類別 1 的比例', fontsize=13)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([f'Fold {i+1}' for i in range(n_splits)])
axes[0].legend(fontsize=10)
axes[0].set_ylim(0, max(max(normal_ratios), max(strat_ratios)) * 1.3)

# 右圖：原始資料分布
for cls, color, label in zip([0, 1], ['#3498db', '#e74c3c'], 
                              ['Class 0 (majority)', 'Class 1 (minority)']):
    mask = y_imb == cls
    axes[1].scatter(X_imb[mask, 0], X_imb[mask, 1], c=color, 
                    alpha=0.6, s=40, edgecolors='white', label=label)

axes[1].set_title('Imbalanced Dataset Distribution\n不平衡資料集分布', fontsize=13)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n觀察重點 Key Observation:")
print(f"  KFold 各 Fold 少數類比例: {[f'{r:.2%}' for r in normal_ratios]}")
print(f"  StratifiedKFold 各 Fold 少數類比例: {[f'{r:.2%}' for r in strat_ratios]}")
print(f"  原始比例 Original ratio: {original_ratio:.2%}")
print("\n  --> StratifiedKFold 確保每個 Fold 的比例接近原始比例！")

## Part 4: 偏差-變異權衡視覺化 Bias-Variance Tradeoff Visualization

### 4.1 用多項式回歸展示不同複雜度的影響

使用不同 degree 的多項式回歸模型擬合同一組資料，觀察偏差與變異的權衡。

In [ ]:
# 建立回歸資料集 Create regression dataset
np.random.seed(42)
n_points = 30

# 真實函數: y = sin(2*pi*x) * x + noise
X_reg = np.sort(np.random.uniform(0, 1, n_points))
y_true_func = lambda x: np.sin(2 * np.pi * x) * x
y_reg = y_true_func(X_reg) + np.random.normal(0, 0.15, n_points)

# 不同 degree 的多項式回歸
degrees = [1, 3, 5, 9, 15]

fig, axes = plt.subplots(1, len(degrees), figsize=(22, 4))
X_plot = np.linspace(0, 1, 200)

for i, degree in enumerate(degrees):
    ax = axes[i]
    
    # 擬合多項式回歸
    poly_model = make_pipeline(
        PolynomialFeatures(degree, include_bias=False),
        LinearRegression()
    )
    poly_model.fit(X_reg.reshape(-1, 1), y_reg)
    y_pred = poly_model.predict(X_plot.reshape(-1, 1))
    
    # 繪製
    ax.scatter(X_reg, y_reg, c='#3498db', s=30, alpha=0.7, 
               edgecolors='white', zorder=3, label='Data')
    ax.plot(X_plot, y_true_func(X_plot), 'g--', alpha=0.5, 
            linewidth=1.5, label='True function')
    ax.plot(X_plot, y_pred, 'r-', linewidth=2, label=f'Degree {degree}')
    
    # 計算訓練與測試 MSE
    y_train_pred = poly_model.predict(X_reg.reshape(-1, 1))
    train_mse = mean_squared_error(y_reg, y_train_pred)
    
    if degree <= 1:
        label = "Underfitting\n(欠擬合)"
    elif degree <= 5:
        label = "Good Fit\n(良好擬合)"
    else:
        label = "Overfitting\n(過擬合)"
    
    ax.set_title(f'Degree = {degree}\n{label}\nTrain MSE = {train_mse:.4f}', 
                 fontsize=11)
    ax.set_xlabel('x')
    if i == 0:
        ax.set_ylabel('y')
    ax.legend(fontsize=7, loc='upper right')
    ax.set_ylim(-1.5, 1.5)

plt.suptitle('Polynomial Regression: Effect of Model Complexity\n多項式回歸：模型複雜度的影響',
             fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

### 4.2 偏差-變異權衡曲線

計算不同 degree 下的訓練誤差與交叉驗證誤差，繪製經典的偏差-變異權衡圖。

In [ ]:
# 偏差-變異權衡曲線 Bias-Variance Tradeoff Curve
# 計算不同 degree 的訓練 MSE 和 CV MSE

degrees_range = range(1, 16)
train_errors = []
cv_errors = []
cv_stds = []

X_reg_2d = X_reg.reshape(-1, 1)

for degree in degrees_range:
    poly_model = make_pipeline(
        PolynomialFeatures(degree, include_bias=False),
        Ridge(alpha=1e-6)  # 輕微正則化避免數值不穩定
    )
    
    # 訓練誤差
    poly_model.fit(X_reg_2d, y_reg)
    y_train_pred = poly_model.predict(X_reg_2d)
    train_mse = mean_squared_error(y_reg, y_train_pred)
    train_errors.append(train_mse)
    
    # CV 誤差（使用負 MSE，scikit-learn 慣例）
    cv_scores = cross_val_score(poly_model, X_reg_2d, y_reg, 
                                 cv=5, scoring='neg_mean_squared_error')
    cv_errors.append(-cv_scores.mean())
    cv_stds.append(cv_scores.std())

# 繪製偏差-變異權衡圖
fig, ax = plt.subplots(figsize=(10, 6))

degrees_list = list(degrees_range)
ax.plot(degrees_list, train_errors, 'b-o', linewidth=2, markersize=6,
        label='Training Error (訓練誤差)', color='#3498db')
ax.plot(degrees_list, cv_errors, 'r-s', linewidth=2, markersize=6,
        label='CV Error (交叉驗證誤差)', color='#e74c3c')

# 填充 CV 誤差的標準差區間
cv_errors_arr = np.array(cv_errors)
cv_stds_arr = np.array(cv_stds)
ax.fill_between(degrees_list, 
                cv_errors_arr - cv_stds_arr, 
                cv_errors_arr + cv_stds_arr,
                alpha=0.15, color='#e74c3c')

# 標註最佳 degree
best_degree = degrees_list[np.argmin(cv_errors)]
best_cv_error = min(cv_errors)
ax.axvline(x=best_degree, color='green', linestyle='--', alpha=0.7)
ax.annotate(f'Best degree = {best_degree}\nCV MSE = {best_cv_error:.4f}',
            xy=(best_degree, best_cv_error),
            xytext=(best_degree + 2, best_cv_error + 0.05),
            fontsize=11, color='green',
            arrowprops=dict(arrowstyle='->', color='green'))

# 標註區域
ax.annotate('Underfitting\n(欠擬合)\nHigh Bias', 
            xy=(1.5, 0.08), fontsize=10, color='gray',
            ha='center', style='italic')
ax.annotate('Overfitting\n(過擬合)\nHigh Variance', 
            xy=(13, 0.08), fontsize=10, color='gray',
            ha='center', style='italic')

ax.set_xlabel('Polynomial Degree (多項式次數)', fontsize=13)
ax.set_ylabel('Mean Squared Error (MSE)', fontsize=13)
ax.set_title('Bias-Variance Tradeoff\n偏差-變異權衡圖', fontsize=15)
ax.legend(fontsize=12)
ax.set_xticks(degrees_list)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

print(f"\n最佳多項式次數 Best polynomial degree: {best_degree}")
print(f"對應 CV MSE: {best_cv_error:.4f}")

### 4.3 多次取樣觀察變異 (Variance)

同一個 degree 的模型，在不同訓練資料子集上的擬合結果差異有多大？這就是「變異」的體現。

In [ ]:
# 用多次 bootstrap 取樣展示偏差 vs 變異
# Show bias vs variance with bootstrap sampling

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
selected_degrees = [1, 4, 15]
titles = ['Degree 1 (High Bias, Low Variance)\n高偏差、低變異',
          'Degree 4 (Balanced)\n偏差與變異平衡',
          'Degree 15 (Low Bias, High Variance)\n低偏差、高變異']

X_plot = np.linspace(0, 1, 200).reshape(-1, 1)
n_bootstrap = 20

for ax_idx, (degree, title) in enumerate(zip(selected_degrees, titles)):
    ax = axes[ax_idx]
    
    # 多次 bootstrap 取樣 + 擬合
    for i in range(n_bootstrap):
        # Bootstrap 取樣
        boot_idx = np.random.choice(n_points, size=n_points, replace=True)
        X_boot = X_reg[boot_idx].reshape(-1, 1)
        y_boot = y_reg[boot_idx]
        
        poly_model = make_pipeline(
            PolynomialFeatures(degree, include_bias=False),
            Ridge(alpha=1e-6)
        )
        poly_model.fit(X_boot, y_boot)
        y_pred = poly_model.predict(X_plot)
        
        ax.plot(X_plot, y_pred, color='#e74c3c', alpha=0.15, linewidth=1)
    
    # 繪製真實函數與資料點
    ax.plot(X_plot.ravel(), y_true_func(X_plot.ravel()), 
            'g--', linewidth=2, label='True function', zorder=5)
    ax.scatter(X_reg, y_reg, c='#3498db', s=25, alpha=0.6, 
               edgecolors='white', zorder=4, label='Data')
    
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('x')
    if ax_idx == 0:
        ax.set_ylabel('y')
    ax.set_ylim(-1.5, 1.5)
    ax.legend(fontsize=8, loc='upper right')

plt.suptitle(f'Bias-Variance Visualization ({n_bootstrap} Bootstrap Samples)\n'
             f'偏差-變異視覺化（{n_bootstrap} 次 Bootstrap 取樣）', 
             fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print("觀察重點 Key Observations:")
print("  Degree 1:  曲線幾乎不隨資料改變（低變異），但系統性偏離真實函數（高偏差）")
print("  Degree 4:  曲線大致追蹤真實函數，波動適中（偏差與變異平衡）")
print("  Degree 15: 曲線隨資料劇烈波動（高變異），但通過訓練點附近（低偏差）")

## Part 5: 過擬合/欠擬合視覺化 Overfitting/Underfitting Visualization

### 5.1 分類問題中的過擬合與欠擬合

使用 Moon 形資料集和不同複雜度的分類模型，觀察決策邊界的變化。

In [ ]:
# 分類問題中的過擬合/欠擬合
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

# 建立 Moon 形資料集
X_moon, y_moon = make_moons(n_samples=200, noise=0.3, random_state=42)
X_moon_train, X_moon_test, y_moon_train, y_moon_test = train_test_split(
    X_moon, y_moon, test_size=0.3, random_state=42
)

# 不同複雜度的模型
models = [
    ("Logistic Regression\n(Linear, 欠擬合)", 
     LogisticRegression(random_state=42)),
    ("Decision Tree (depth=3)\n(適當擬合)", 
     DecisionTreeClassifier(max_depth=3, random_state=42)),
    ("Decision Tree (depth=None)\n(過擬合)", 
     DecisionTreeClassifier(max_depth=None, random_state=42)),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, model) in enumerate(models):
    ax = axes[idx]
    
    # 訓練模型
    model.fit(X_moon_train, y_moon_train)
    train_acc = model.score(X_moon_train, y_moon_train)
    test_acc = model.score(X_moon_test, y_moon_test)
    
    # 繪製決策邊界
    h = 0.02
    x_min, x_max = X_moon[:, 0].min() - 0.5, X_moon[:, 0].max() + 0.5
    y_min, y_max = X_moon[:, 1].min() - 0.5, X_moon[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                          np.arange(y_min, y_max, h))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#3498db', '#e74c3c']))
    ax.contour(xx, yy, Z, colors='black', linewidths=0.5, alpha=0.3)
    
    # 繪製訓練與測試資料
    for cls, color in zip([0, 1], ['#2980b9', '#c0392b']):
        mask_train = y_moon_train == cls
        mask_test = y_moon_test == cls
        ax.scatter(X_moon_train[mask_train, 0], X_moon_train[mask_train, 1],
                   c=color, s=30, alpha=0.6, edgecolors='white', label=f'Train cls {cls}')
        ax.scatter(X_moon_test[mask_test, 0], X_moon_test[mask_test, 1],
                   c=color, s=60, alpha=0.9, marker='x', linewidths=2,
                   label=f'Test cls {cls}')
    
    gap = train_acc - test_acc
    ax.set_title(f'{name}\nTrain: {train_acc:.3f} | Test: {test_acc:.3f} | Gap: {gap:.3f}',
                 fontsize=10)
    ax.legend(fontsize=7, loc='upper right')
    ax.set_xlabel('Feature 1')
    if idx == 0:
        ax.set_ylabel('Feature 2')

plt.suptitle('Underfitting vs Good Fit vs Overfitting (Classification)\n'
             '欠擬合 vs 良好擬合 vs 過擬合（分類問題）', 
             fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print("\n診斷指引 Diagnosis Guide:")
print("  Train-Test Gap 小 + 兩者都低 → 欠擬合（模型太簡單）")
print("  Train-Test Gap 適中 + 兩者都合理 → 良好擬合")
print("  Train-Test Gap 大 + 訓練高測試低 → 過擬合（模型太複雜）")

### 5.2 時間序列分割視覺化 Time Series Split Visualization

觀察 `TimeSeriesSplit` 如何確保訓練集始終在驗證集之前。

In [ ]:
# 時間序列分割視覺化 Time Series Split Visualization

n_samples_ts = 60
n_splits_ts = 5
tscv = TimeSeriesSplit(n_splits=n_splits_ts)
indices_ts = np.arange(n_samples_ts)

fig, axes = plt.subplots(n_splits_ts, 1, figsize=(14, 6), sharex=True)

for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(indices_ts)):
    ax = axes[fold_idx]
    
    # 每個樣本的顏色：灰色=未使用, 藍色=訓練, 紅色=驗證
    colors_ts = np.array(['#ecf0f1'] * n_samples_ts)
    colors_ts[train_idx] = '#3498db'
    colors_ts[val_idx] = '#e74c3c'
    
    for i in range(n_samples_ts):
        ax.barh(0, 1, left=i, color=colors_ts[i], edgecolor='white', linewidth=0.3)
    
    ax.set_yticks([0])
    ax.set_yticklabels([f'Split {fold_idx + 1}'], fontsize=11)
    ax.set_xlim(0, n_samples_ts)
    ax.set_ylim(-0.5, 0.5)
    
    # 標註
    ax.annotate(f'Train: {len(train_idx)}', 
                xy=(train_idx[len(train_idx)//2], 0.3),
                fontsize=8, ha='center', color='#2980b9', fontweight='bold')
    ax.annotate(f'Val: {len(val_idx)}', 
                xy=(val_idx[len(val_idx)//2], 0.3),
                fontsize=8, ha='center', color='#c0392b', fontweight='bold')

axes[-1].set_xlabel('Time Index (時間索引)', fontsize=12)

# 圖例
unused_patch = mpatches.Patch(color='#ecf0f1', label='Unused (未使用)')
train_patch = mpatches.Patch(color='#3498db', label='Training (訓練)')
val_patch = mpatches.Patch(color='#e74c3c', label='Validation (驗證)')
fig.legend(handles=[unused_patch, train_patch, val_patch], 
           loc='upper right', fontsize=10)

plt.suptitle('TimeSeriesSplit Visualization\n時間序列分割視覺化', 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("觀察重點 Key Observation:")
print("  1. 訓練集始終在驗證集之前（時間順序）")
print("  2. 訓練集逐步擴大（累積更多歷史資料）")
print("  3. 不會發生「用未來資料預測過去」的情況")

## Part 6: 練習題 Exercises

完成以下練習以鞏固本週學習內容。

---

### 練習 1：資料分割與評估

載入 scikit-learn 的 Iris 資料集，進行以下操作：
1. 使用 `train_test_split` 分割資料（70% 訓練、30% 測試，使用分層抽樣）
2. 訓練一個 `LogisticRegression` 模型
3. 分別計算訓練集與測試集的準確率
4. 判斷模型是否存在過擬合或欠擬合

In [ ]:
# 練習 1：請在此完成你的程式碼
# Exercise 1: Complete your code here

from sklearn.datasets import load_iris

# 載入資料
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

# TODO: 1. 使用 train_test_split 分割資料（test_size=0.3, stratify=y_iris, random_state=42）
# X_train_iris, X_test_iris, y_train_iris, y_test_iris = ...

# TODO: 2. 訓練 LogisticRegression 模型
# model_iris = ...

# TODO: 3. 計算訓練集與測試集準確率
# train_acc = ...
# test_acc = ...

# TODO: 4. 判斷是否過擬合/欠擬合
# print(f"訓練準確率: {train_acc:.4f}")
# print(f"測試準確率: {test_acc:.4f}")
# print(f"Gap: {train_acc - test_acc:.4f}")
# 根據 gap 大小判斷模型狀態

### 練習 2：交叉驗證比較

使用 Iris 資料集，比較以下三種交叉驗證策略：
1. 5-Fold KFold
2. 5-Fold StratifiedKFold
3. 10-Fold StratifiedKFold

繪製 boxplot 比較三者的分數分布。

In [ ]:
# 練習 2：請在此完成你的程式碼
# Exercise 2: Complete your code here

# TODO: 建立三種 CV 策略並計算 cross_val_score
# cv1 = KFold(n_splits=5, shuffle=True, random_state=42)
# cv2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# cv3 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# model = LogisticRegression(max_iter=200, random_state=42)
# scores1 = cross_val_score(model, X_iris, y_iris, cv=cv1, scoring='accuracy')
# scores2 = ...
# scores3 = ...

# TODO: 繪製 boxplot 比較
# fig, ax = plt.subplots(figsize=(8, 5))
# ax.boxplot([scores1, scores2, scores3], labels=['5-Fold KF', '5-Fold SKF', '10-Fold SKF'])
# ...

### 練習 3：偏差-變異權衡挑戰

使用 `make_regression` 建立一個有雜訊的回歸問題，然後：
1. 嘗試 degree = 1, 2, 3, 5, 7, 10, 15 的多項式回歸
2. 用 5-Fold CV 計算每個 degree 的平均 MSE
3. 找出最佳 degree 並繪製對應的擬合曲線
4. 在同一張圖上繪製訓練誤差和 CV 誤差隨 degree 的變化

In [ ]:
# 練習 3：請在此完成你的程式碼
# Exercise 3: Complete your code here

from sklearn.datasets import make_regression

# TODO: 建立回歸資料集
# X_ex, y_ex = make_regression(n_samples=100, n_features=1, noise=20, random_state=42)

# TODO: 嘗試不同 degree 的多項式回歸
# degrees_ex = [1, 2, 3, 5, 7, 10, 15]
# train_errors_ex = []
# cv_errors_ex = []
# for degree in degrees_ex:
#     ...

# TODO: 繪製偏差-變異權衡圖
# ...

---

## 本週總結 Summary

### 關鍵收穫 Key Takeaways

1. **資料分割是評估模型泛化能力的基礎** -- 永遠不要用訓練資料評估模型
2. **交叉驗證提供更穩定的評估** -- 多次分割取平均，降低隨機性影響
3. **分層抽樣在類別不平衡時至關重要** -- 確保每個 Fold 的類別比例一致
4. **偏差-變異權衡是模型選擇的核心思維** -- 太簡單 → 欠擬合，太複雜 → 過擬合
5. **注意資料洩漏** -- 前處理必須在分割之後（或 CV 內部）進行
6. **時間序列需要特殊處理** -- 不能用未來資料預測過去

### 下週預告 Next Week Preview

**第 4 週：線性回歸 -- 損失函數、梯度下降視覺化**
- 將使用本週學到的 CV 技術來評估回歸模型